# Historic ECMWF PF (TIGGE ensemble via cdsapi) - reconstruccion historica

Reconstruye el historico del ensemble perturbado (`pf`, 50 miembros) del dataset `tigge-forecasts`, pidiendo **un mes calendario por request** (no un anio, a diferencia de `Historic_ECMWF_CF`): con 50 miembros, un lote anual tendria ~292.000 "fields" (365 dias x 16 steps x 50 miembros), un orden de magnitud grande y arriesgado para un solo request. Un lote mensual da ~24.000 fields, comparable al limite documentado de otros datasets CDS (ERA5 horario: 120.000). El archivo TIGGE arranca en 2006-10-01. Escribe un JSON por dia en el mismo folder que usa el job diario (`Daily_ECMWF_PF`), asi Bronze no requiere ningun cambio.

**Reglas de seguridad frente a la API (no negociables):** un request a la vez, tope de `max_batches_per_run` lotes por corrida, corte inmediato ante el primer fallo (no reintentar en bucle), resumible por diseno (saltea lotes ya completos). Este notebook corre en un job separado, sin schedule, que nunca debe superponerse en el tiempo con `ECMWF_Forecast_Daily_Incremental` ni con `Historic_ECMWF_CF` (comparten cuenta/token con la misma cola de TIGGE/ECDS).

In [ ]:
%pip install --quiet cdsapi geopandas pyogrio xarray netCDF4
dbutils.library.restartPython()

In [ ]:
import json
import math
import time
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

import geopandas as gpd

GRID_DEG = 0.25
GEOJSON_PATH = "/Workspace/Users/joaquintschopp@gmail.com/rio-uruguay-hydro-pipeline/SIG/subcuencas_modelo.geojson"


def compute_download_area(geojson_path=GEOJSON_PATH, grid_deg=GRID_DEG, margin_cells=1):
    gdf = gpd.read_file(geojson_path)
    minx, miny, maxx, maxy = gdf.total_bounds
    margin = grid_deg * margin_cells
    north = math.ceil((maxy + margin) / grid_deg) * grid_deg
    south = math.floor((miny - margin) / grid_deg) * grid_deg
    west = math.floor((minx - margin) / grid_deg) * grid_deg
    east = math.ceil((maxx + margin) / grid_deg) * grid_deg
    return {"north": round(north, 4), "west": round(west, 4), "south": round(south, 4), "east": round(east, 4)}


def area_to_cds_list(area):
    return [area["north"], area["west"], area["south"], area["east"]]


def normalize_longitude(lon):
    return lon - 360 if lon > 180 else lon


def point_in_bbox(lat, lon, area):
    lon_norm = normalize_longitude(lon)
    return (area["south"] <= lat <= area["north"]) and (area["west"] <= lon_norm <= area["east"])


def raw_filename(tipo, run_date, run_time, ext):
    return f"ECMWF_{tipo.upper()}_{run_date:%Y_%m_%d}_t{run_time}.{ext}"


def already_landed(tipo, run_date, run_time, json_dir):
    return (json_dir / raw_filename(tipo, run_date, run_time, "json")).exists()


def days_in_range(start, end):
    n = (end - start).days
    for i in range(n + 1):
        yield start + timedelta(days=i)


def batch_fully_landed(tipo, start, end, run_time, json_dir):
    return all(already_landed(tipo, d, run_time, json_dir) for d in days_in_range(start, end))


def date_range_str(start, end):
    # Sintaxis del portal ECMWF Data Stores (ecds.ecmwf.int): "start/end", sin "/to/"
    # (esa era sintaxis MARS clasica; la API nueva la rechaza con 400 Bad Request).
    return f"{start.isoformat()}/{end.isoformat()}"


def iter_batches_backward(earliest, latest, step_months):
    """Lotes [start, end] desde el mas reciente hacia el mas antiguo, de tamanio step_months,
    acotados a [earliest, latest]. El lote mas antiguo queda pegado a 'earliest' sin resto suelto."""
    from dateutil.relativedelta import relativedelta

    batches = []
    end = latest
    while end >= earliest:
        start = end - relativedelta(months=step_months) + timedelta(days=1)
        if start < earliest:
            start = earliest
        batches.append((start, end))
        end = start - timedelta(days=1)
    return batches


def _step_to_hours(step_val):
    if step_val is None:
        return 0
    import numpy as np
    if isinstance(step_val, np.timedelta64):
        return int(step_val / np.timedelta64(1, "h"))
    return int(step_val)


def _compute_valid_datetime(run_date, run_time, step_hours):
    run_dt = datetime.combine(run_date, datetime.strptime(run_time, "%H").time(), tzinfo=timezone.utc)
    return run_dt + timedelta(hours=step_hours)


def _reftime_dim_name(tp):
    known = {"step", "number", "latitude", "longitude"}
    for d in tp.dims:
        if d not in known:
            return d
    return None


def _reftime_to_run_date_time(value):
    import numpy as np
    ts = value if not isinstance(value, np.datetime64) else value.astype("datetime64[s]").item()
    return ts.date(), f"{ts.hour:02d}"


def flatten_ensemble_forecast_batch(ds, tipo, source_api, unit_to_mm_factor, area=None):
    """Igual que flatten_forecast_batch, pero recorriendo tambien la dimension real 'number'
    (50 miembros del ensemble), para lotes historicos de pf. Devuelve {run_date.isoformat(): records}."""
    tp = ds["tp"]
    lats = ds["latitude"].values
    lons = ds["longitude"].values
    numbers = ds["number"].values

    reftime_dim = _reftime_dim_name(tp)
    reftimes = ds[reftime_dim].values
    steps = ds["step"].values if "step" in tp.dims else [None]

    out = {}
    for rt_idx, rt_val in enumerate(reftimes):
        run_date, run_time = _reftime_to_run_date_time(rt_val)
        tp_day = tp.isel({reftime_dim: rt_idx})
        extracted_at = datetime.now(timezone.utc).isoformat()
        records = []
        for number_idx, number_val in enumerate(numbers):
            tp_member = tp_day.isel(number=number_idx)
            for step_idx, step_val in enumerate(steps):
                step_hours = _step_to_hours(step_val)
                valid_dt = _compute_valid_datetime(run_date, run_time, step_hours)
                slice_2d = tp_member.isel(step=step_idx).values if "step" in tp_member.dims else tp_member.values
                for i, lat in enumerate(lats):
                    for j, lon in enumerate(lons):
                        lon_norm = normalize_longitude(float(lon))
                        if area is not None and not point_in_bbox(float(lat), lon_norm, area):
                            continue
                        value = slice_2d[i, j]
                        if value is None:
                            continue
                        value_f = float(value)
                        if math.isnan(value_f):
                            continue
                        records.append({
                            "run_date": run_date.isoformat(), "run_time": run_time, "step_hours": int(step_hours),
                            "valid_datetime": valid_dt.isoformat(), "valid_date": valid_dt.date().isoformat(),
                            "latitude": float(lat), "longitude": lon_norm, "number": int(number_val),
                            "tp_mm": value_f * unit_to_mm_factor,
                            "tipo": tipo, "source_api": source_api, "extracted_at": extracted_at,
                        })
        out[run_date.isoformat()] = records
    return out


def write_json(records, out_path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(records, ensure_ascii=False), encoding="utf-8")

In [ ]:
try:
    dbutils.widgets.text("max_batches_per_run", "3")
    dbutils.widgets.dropdown("force_reload", "false", ["false", "true"])
    max_batches_per_run = int(dbutils.widgets.get("max_batches_per_run"))
    force_reload = dbutils.widgets.get("force_reload").lower() == "true"
except Exception:
    max_batches_per_run = 3
    force_reload = False

RAW_DIR = Path("/Volumes/weather/raw/ecmwf_volume/pf_tigge/raw/historic")
JSON_DIR = Path("/Volumes/weather/raw/ecmwf_volume/pf_tigge/json")  # mismo folder que el job diario

DATASET = "tigge-forecasts"
ORIGIN = "ecmf"
PARAM = "228228"
STEPS_HOURS = list(range(0, 361, 24))
MEMBERS = list(range(1, 51))
RUN_TIME = "00"

EARLIEST_TIGGE_DATE = date(2006, 10, 1)  # inicio real del archivo TIGGE, no hay datos antes
TIGGE_LAG_DAYS = 2  # mismo margen que el job diario, para no pisar su ventana
BATCH_MONTHS = 1  # 1 request = 1 mes calendario (50 miembros multiplican los fields)
UNIT_TO_MM_FACTOR = 1.0
PAUSE_BETWEEN_REQUESTS_SECONDS = 2

print(f"max_batches_per_run={max_batches_per_run}, force_reload={force_reload}")

In [ ]:
cdsapi_url = dbutils.secrets.get(scope="ecmwf", key="cdsapi_url")
cdsapi_key = dbutils.secrets.get(scope="ecmwf", key="cdsapi_key")

In [ ]:
import cdsapi
import xarray as xr


def _batch_raw_path(start, end):
    return RAW_DIR / f"ECMWF_PF_{start.isoformat()}_{end.isoformat()}.nc"


def _retrieve_batch(client, start, end, area, target):
    request = {
        "origin": ORIGIN, "levtype": "sfc", "param": PARAM, "type": "pf",
        "number": [str(n) for n in MEMBERS],
        "step": [str(s) for s in STEPS_HOURS], "date": date_range_str(start, end), "time": "00:00:00",
        "area": area_to_cds_list(area), "grid": [0.25, 0.25], "format": "netcdf",
    }
    try:
        client.retrieve(DATASET, request, str(target))
        return True
    except Exception as e:
        print(f"  FALLO lote {start.isoformat()}..{end.isoformat()}: {str(e)[:300]}")
        return False


latest = date.today() - timedelta(days=TIGGE_LAG_DAYS)
batches = iter_batches_backward(EARLIEST_TIGGE_DATE, latest, BATCH_MONTHS)
print(f"Rango objetivo: {EARLIEST_TIGGE_DATE.isoformat()} .. {latest.isoformat()} ({len(batches)} lotes mensuales totales)")

client = cdsapi.Client(url=cdsapi_url, key=cdsapi_key)
area = compute_download_area()
print(f"Area de descarga calculada: {area}")
RAW_DIR.mkdir(parents=True, exist_ok=True)

processed = 0
written_dates = set()
for start, end in batches:
    if processed >= max_batches_per_run:
        print(f"Limite de {max_batches_per_run} lotes por corrida alcanzado, se corta aca. Volver a correr para continuar.")
        break

    if not force_reload and batch_fully_landed("pf", start, end, RUN_TIME, JSON_DIR):
        print(f"Lote {start.isoformat()}..{end.isoformat()} ya completo, skip")
        continue

    print(f"Pidiendo lote {start.isoformat()}..{end.isoformat()} ({(end - start).days + 1} dias x {len(MEMBERS)} miembros)...")
    raw_path = _batch_raw_path(start, end)
    if not _retrieve_batch(client, start, end, area, raw_path):
        print("Se corta la ejecucion por el fallo anterior (no se reintenta en bucle).")
        break

    ds = xr.open_dataset(raw_path, engine="netcdf4", decode_timedelta=False)
    by_day = flatten_ensemble_forecast_batch(ds, tipo="pf", source_api="ecmwf_tigge_cdsapi_historic", unit_to_mm_factor=UNIT_TO_MM_FACTOR, area=None)
    for run_date_iso, records in by_day.items():
        json_path = JSON_DIR / raw_filename("pf", date.fromisoformat(run_date_iso), RUN_TIME, "json")
        write_json(records, json_path)
        written_dates.add(run_date_iso)
    print(f"OK lote {start.isoformat()}..{end.isoformat()}: {len(by_day)} dias, {sum(len(r) for r in by_day.values())} registros")

    processed += 1
    time.sleep(PAUSE_BETWEEN_REQUESTS_SECONDS)

if processed == 0:
    print("Nada pendiente para procesar en esta corrida (o limite en 0). Si processed==0 y no queda nada PENDIENTE, el backfill esta completo.")

# Publica el rango efectivamente escrito en esta corrida como task value, para que el
# task de ETL_Silver_ECMWF_PF (load_mode=backfill) en el job de backfill lo consuma via
# {{tasks.Historic_ECMWF_PF.values.range_start/range_end}} sin tener que hardcodear fechas
# en databricks.yml. Si no se escribio nada, se publica vacio y Silver simplemente sale.
try:
    dbutils.jobs.taskValues.set(key="range_start", value=min(written_dates) if written_dates else "")
    dbutils.jobs.taskValues.set(key="range_end", value=max(written_dates) if written_dates else "")
except Exception:
    pass